# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`
This notebook guides you in loading and exploring the FAIR^2 dataset using the `mlcroissant` library. We'll review available record sets and fields, extract and process tabular data, and provide basic exploratory analysis steps.

### Dataset Source
The dataset is defined via a Croissant schema accessible at:
[https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

In [ ]:
# Ensure mlcroissant is installed
!pip install -q mlcroissant

## 1. Data Loading
We load metadata and records with `mlcroissant`, and inspect general dataset metadata.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access the high-level metadata
meta = dataset.metadata
print(f"{meta.name}: {meta.description}\n")
print(f"Dataset identifier (DOI): {getattr(meta, 'identifier', 'N/A')}")
print(f"Authors: {[a.get('@id', str(a)) for a in getattr(meta, 'author', [])]}")

## 2. Data Overview
Review available record sets, their `@id`s, and contained fields. All entity references use their `@id` as per Croissant best practice.

In [ ]:
# List all record sets present in the dataset schema
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets defined in schema. The dataset may only provide document-level metadata or lacks machine-readable tabular data.")
else:
    print(f"Record sets available ({len(record_sets)}):")
    for rs in record_sets:
        print(f"  - @id: {rs.id}")
        print(f"    name: {rs.name}")
        print("    fields:")
        for fld in rs.fields:
            print(f"      * @id: {fld.id} | name: {fld.name} | type: {fld.data_type}")

# For demonstration, try to print the first 2 records for each record set
for rs in record_sets:
    print(f"\nRecords from record set @id: {rs.id}")
    for i, rec in enumerate(dataset.records(record_set=rs.id)):
        print(rec)
        if i >= 1:
            break

## 3. Data Extraction
Load records from each record set (referenced by `@id`) into pandas DataFrames for further analysis.

- List all record set `@id`s found above.
- DataFrame column names are the field ids.

In [ ]:
dataframes = {}
all_rs_ids = [rs.id for rs in record_sets] if record_sets else []
for rs_id in all_rs_ids:
    recs = list(dataset.records(record_set=rs_id))
    if recs:
        df = pd.DataFrame(recs)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records from record set '@id': {rs_id}")
        print(f"Columns: {df.columns.tolist()}")
        display(df.head(3))
    else:
        print(f"No records found for record set '@id': {rs_id}")

# If no record sets or DataFrames, fallback: inform the user
if not dataframes:
    print("No tabular data was found in the published schema. Only metadata is available.")

## 4. Exploratory Data Analysis (EDA)
If record sets are available, perform simple exploratory analytics:
- Filtering numeric fields by threshold
- Normalizing a field
- Grouping by a categorical field

> All columns and fields are referenced by their `@id`.

In [ ]:
if dataframes:
    # Use the first record set
    primary_rs_id = list(dataframes.keys())[0]
    df = dataframes[primary_rs_id]
    print(f"Performing EDA on record set '@id': {primary_rs_id}")
    # Infer numeric columns for demonstration
    numeric_columns = df.select_dtypes(include='number').columns
    if len(numeric_columns) == 0:
        print("No numeric fields detected for EDA in this record set.")
    else:
        # Pick the first numeric field
        numeric_field_id = numeric_columns[0]
        print(f"Using numeric field '@id': {numeric_field_id}")
        # Set an example threshold: 10 or 0 if no large values
        threshold = 10 if df[numeric_field_id].max() > 10 else 0
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where {numeric_field_id} > {threshold}: {len(filtered_df)} records.")
        display(filtered_df.head())

        # Normalize
        col_normalized = f"{numeric_field_id}_normalized"
        filtered_df[col_normalized] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, col_normalized]].head())

        # Attempt grouping by a non-numeric column if present
        group_fields = [col for col in df.columns if col != numeric_field_id and df[col].nunique() <= df.shape[0]//2]
        if group_fields:
            group_field_id = group_fields[0]
            print(f"Grouping statistics by field '@id': {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            display(grouped_df.head())
        else:
            print("No suitable categorical/group fields found for grouping.")
else:
    print("EDA skipped: no record sets/dataframes are loaded.")

## 5. Visualization
If extracted, plot field distributions or group comparisons. Uses matplotlib/seaborn for simple plots.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    df = dataframes[list(dataframes.keys())[0]]
    numeric_columns = df.select_dtypes(include='number').columns
    if len(numeric_columns) > 0:
        col = numeric_columns[0]
        plt.figure(figsize=(7,3))
        sns.histplot(df[col], kde=True, bins=15)
        plt.xlabel(col)
        plt.title(f"Distribution of '{col}' (@id)")
        plt.show()

        # Grouped boxplot by group field if present
        possible_group_cols = [c for c in df.columns if c != col and df[c].nunique() < df.shape[0]//2]
        if possible_group_cols:
            group_col = possible_group_cols[0]
            plt.figure(figsize=(6,4))
            sns.boxplot(x=df[group_col].astype(str), y=df[col])
            plt.xlabel(group_col)
            plt.ylabel(col)
            plt.title(f'Boxplot of {col} grouped by {group_col}')
            plt.show()
    else:
        print("No numeric columns to visualize.")
else:
    print("Visualization skipped: no data to plot.")

## 6. Conclusion
- We loaded the FAIR^2 dataset schema using `mlcroissant`, explored metadata and available record sets, and, if provided, extracted and previewed tabular data referenced by their `@id`.
- Exploratory steps were performed using field `@id` references exclusively. 
- For production analysis, use precise field `@id` from the Croissant schema (see section 2).
- Note: if the dataset lacks record sets or tabular data, only metadata is accessible in a machine-readable way.